# Day of the Race — Solution
## Analyze Data with R: Vectors, Lists, Functions & Apply Family

Complete working solution with alternate implementations, extra practice answers, and a parameterised simulation.

![Flowchart](day_of_the_race_flowchart.png)


## 0. Reference Data

In [ ]:
race_results <- c("Gi", "Francesca", "Lea", "Vivian", "Jessica",
                  "Esther", "Mary", "Yasmina", "Megan", "Janet",
                  "Tiffany", "Kishan", "Feng", "Z", "Tina")

info_list <- list(
  Esther = list(jersey = 3432, color = "purple"),
  Feng   = list(jersey = 4221, color = "blue")
)

cat("Race results length:", length(race_results), "\n")
print(race_results)


## 1. friends vector

In [ ]:
friends <- c("Megan", "Janet", "Tina")
print(friends)
# Expected: [1] "Megan" "Janet" "Tina"


## 2. Extend info_list

In [ ]:
info_list[["Megan"]] <- list(jersey = 1363, color = "green")
info_list[["Janet"]] <- list(jersey = 6729, color = "green")
info_list[["Tina"]]  <- list(jersey = 7501, color = "orange")

# Verify
str(info_list)
cat("\nMegan's info:\n")
print(info_list[["Megan"]])


## 3–4. print_information (with loop)

In [ ]:
print_information <- function(name) {
  print(paste(name, "is #", info_list[[name]]$jersey,
              "wearing the color", info_list[[name]]$color))
}

# Individual calls (original requirement)
print_information("Megan")
print_information("Janet")
print_information("Tina")

cat("\n--- Loop version ---\n")
for (f in friends) {
  print_information(f)
}

# Expected output (approx):
# [1] "Megan is # 1363 wearing the color green"
# [1] "Janet is # 6729 wearing the color green"
# [1] "Tina is # 7501 wearing the color orange"


## 5–10. find_place() with proper not-found handling
Returns the 1-based place, or `length(race_results)+1` when the name is absent.


In [ ]:
find_place <- function(runner) {
  for (place in 1:length(race_results)) {
    if (race_results[place] == runner) {
      return(place)
    }
  }
  return(length(race_results) + 1)   # after last place
}

# Tests
cat("Francesca ->", find_place("Francesca"), "(expected 2)\n")
cat("Tina      ->", find_place("Tina"),      "(expected 15)\n")
cat("Gi        ->", find_place("Gi"),        "(expected 1)\n")
cat("Owen      ->", find_place("Owen"),      "(expected 16)\n")


## 11–12. lapply & sapply on friends

In [ ]:
cat("lapply (list):\n")
print(lapply(friends, find_place))

cat("\nsapply (simplified vector):\n")
print(sapply(friends, find_place))

# Expected sapply:
# Megan Janet  Tina
#     9    10    15


## Alternate Implementations
Idiomatic R alternatives that produce the same results.


In [ ]:
# A. match()
find_place_match <- function(runner) {
  idx <- match(runner, race_results)
  if (is.na(idx)) length(race_results) + 1L else as.integer(idx)
}

# B. which()
find_place_which <- function(runner) {
  idx <- which(race_results == runner)
  if (length(idx) == 0) length(race_results) + 1L else idx[1]
}

# C. Named lookup (fast for repeated queries)
place_lookup <- setNames(seq_along(race_results), race_results)
find_place_named <- function(runner) {
  p <- place_lookup[[runner]]          # [[ ]] returns NULL if missing
  if (is.null(p)) length(race_results) + 1L else p
}

# Verify all agree
test_names <- c(friends, "Owen", "Francesca")
data.frame(
  name   = test_names,
  loop   = sapply(test_names, find_place),
  match  = sapply(test_names, find_place_match),
  which  = sapply(test_names, find_place_which),
  named  = sapply(test_names, find_place_named)
)


## More Practice — Solutions

In [ ]:
# 1. Friends wearing a given color
friends_in_color <- function(color) {
  friends[sapply(friends, function(f) info_list[[f]]$color == color)]
}
cat("Green friends:", friends_in_color("green"), "\n")
cat("Orange friends:", friends_in_color("orange"), "\n")

# 2. Data frame of friends
places <- sapply(friends, find_place)
friends_df <- data.frame(
  name   = friends,
  jersey = sapply(friends, function(f) info_list[[f]]$jersey),
  color  = sapply(friends, function(f) info_list[[f]]$color),
  place  = places,
  stringsAsFactors = FALSE
)
print(friends_df)

# 3. Highest finisher among friends (smallest place)
best_friend <- friends_df$name[which.min(friends_df$place)]
cat("\nHighest-finishing friend:", best_friend,
    "at place", min(friends_df$place), "\n")


## Simulation — Monte-Carlo finishing places
We generate many random race orderings and record where our three friends finish.  
Change `n_runners` and `n_sims` to explore sensitivity.


In [ ]:
set.seed(42)
n_runners <- 30
n_sims    <- 800
friend_names <- c("Megan", "Janet", "Tina")

# Simulate: each friend is equally likely to finish in any position 1..n_runners
# (independent for simplicity; a full permutation model is also possible)
place_mat <- matrix(
  sample(1:n_runners, size = n_sims * length(friend_names), replace = TRUE),
  nrow = n_sims, ncol = length(friend_names)
)
colnames(place_mat) <- friend_names

# Summary statistics
cat("Summary of simulated places (n_runners =", n_runners, "):\n")
print(summary(place_mat))

# Empirical probability of finishing in top-10
top10_prob <- colMeans(place_mat <= 10)
cat("\nP(place <= 10):\n")
print(round(top10_prob, 3))

# Quick visual (base R)
# par(mfrow = c(1, 3))
# for (i in seq_along(friend_names)) {
#   hist(place_mat[, i], breaks = 15, main = friend_names[i],
#        xlab = "Place", col = "steelblue", border = "white")
# }


## Key Results Recap
| Friend | Jersey | Color  | Place |
|--------|--------|--------|-------|
| Megan  | 1363   | green  | 9     |
| Janet  | 6729   | green  | 10    |
| Tina   | 7501   | orange | 15    |

Unknown runner (e.g. Owen) → 16 (one past last).

`sapply` produces a clean named numeric vector of places; `lapply` produces a list of the same values.
